# D11 — Kernel-Verified Topological Band Theory & Metamaterial Substrate (Technical)

Companion to `papers/D11/paper_draft.tex`.

**What this notebook is for.** Every quantity below is *also* proved in Lean 4. The point of
re-computing it in Python is not to establish it — the kernel already did — but to give a
numerical cross-check that the Python formula layer in `src/core/formulas.py` agrees with the
Lean statements it claims to mirror. A disagreement here means the Python side has drifted and
is a bug, not a discovery.

**Two-layer honesty.** The algebra below is kernel-verified. The identification of any of it
with a physical material is a literature-cited hypothesis, never smuggled in.

In [1]:
import sys, pathlib
if str(pathlib.Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np
from src.core import formulas as F
from src.core import visualizations as V
print("substrate imported")

substrate imported


## 1. Phononic band gap (Phase 6CB)

Lean certifies, for the diatomic chain `(m₁,m₂,κ,a) = (1,2,1,1)`:

- `phononic_band_gap_exists` — a gap in ω² bracketing both branches for all k
- `branchMinus_at_pi` = 1 and `branchPlus_at_pi` = 2 — **both edges attained**, so the gap is tight
- `acoustic_le_one`, `optical_ge_two` — the global bounds
- `band_gap_rational_enclosure` — the frequency gap by a rational inner bracket, `norm_num`, no floats

In [2]:
lo, hi = F.phononic_gap_edges()
at_pi  = F.diatomic_branches(1.0, 2.0, 1.0, np.pi)
ks     = np.linspace(-np.pi, np.pi, 4001)
br     = np.array([F.diatomic_branches(1.0, 2.0, 1.0, k) for k in ks])

print(f"certified gap edges            : ({lo:g}, {hi:g})")
print(f"branches at k = pi             : ({at_pi[0]:.12g}, {at_pi[1]:.12g})   [Lean: (1, 2)]")
print(f"max acoustic over the BZ       : {br[:,0].max():.12g}   [Lean acoustic_le_one: <= 1]")
print(f"min optical  over the BZ       : {br[:,1].min():.12g}   [Lean optical_ge_two : >= 2]")

assert abs(at_pi[0] - 1.0) < 1e-12 and abs(at_pi[1] - 2.0) < 1e-12
assert br[:,0].max() <= lo + 1e-12 and br[:,1].min() >= hi - 1e-12
print("\nOK - Python reproduces every Lean band-gap value exactly.")

certified gap edges            : (1, 2)
branches at k = pi             : (1, 2)   [Lean: (1, 2)]
max acoustic over the BZ       : 1   [Lean acoustic_le_one: <= 1]
min optical  over the BZ       : 2   [Lean optical_ge_two : >= 2]

OK - Python reproduces every Lean band-gap value exactly.


### The gap is stated as a *refutation*, not an unmet bound

`band_gap_falsifier` says any real ω annihilating the secular determinant strictly inside the gap
yields `False`. Numerically: no sampled ω² lands in the open interval.

In [3]:
inside = ((br[:,0] > lo + 1e-12) & (br[:,0] < hi - 1e-12)) | \
         ((br[:,1] > lo + 1e-12) & (br[:,1] < hi - 1e-12))
print(f"sampled points with a branch strictly inside the gap: {inside.sum()}  (expect 0)")
assert inside.sum() == 0

enc_lo, enc_hi = F.phononic_gap_rational_enclosure()
print(f"rational enclosure of the frequency gap : ({enc_lo:g}, {enc_hi:g})")
print(f"true sqrt(2)                            : {np.sqrt(2):.6f}")
assert enc_hi <= np.sqrt(2.0)
print("OK - the rational bracket is an INNER bracket of the true gap.")

sampled points with a branch strictly inside the gap: 0  (expect 0)
rational enclosure of the frequency gap : (1, 1.41)
true sqrt(2)                            : 1.414214
OK - the rational bracket is an INNER bracket of the true gap.


In [4]:
# viz-ref: fig_d11_phononic_band_gap
V.fig_d11_phononic_band_gap().show()

## 2. The PT transition and the exceptional point (Phase 6CD)

`pt_symmetric_real_spectrum_iff` is a **sharp biconditional**: a real eigenvalue exists iff `g² ≤ 1`.
`exceptional_point_defective` certifies that `g = 1` is genuinely defective (algebraic multiplicity 2,
geometric multiplicity 1) — established by comparing two explicitly computed eigenspaces rather than
by a Jordan normal form, which Mathlib does not have.

In [5]:
for g in (0.0, 0.5, 0.99, 1.0):
    print(f"  g = {g:<5} splitting = {F.pt_eigenvalue_splitting(g):.6f}")
assert F.pt_eigenvalue_splitting(1.0) == 0.0

# ep_proximity_enclosure: 99/100 <= g^2 <= 1  =>  splitting <= 1/5
g2 = np.linspace(0.99, 1.0, 500)
worst = max(F.pt_eigenvalue_splitting(float(np.sqrt(x))) for x in g2)
print(f"\nmax splitting on 99/100 <= g^2 <= 1 : {worst:.12f}   [Lean bound: <= 1/5]")
# The bound is ATTAINED at the left endpoint: Delta(g) = 2*sqrt(1 - 0.99) = 2*0.1 = 1/5 exactly.
# So this is a tight enclosure, not a loose one - compare only up to float epsilon.
assert worst <= 0.2 + 1e-12
print(f"attained at g^2 = 99/100 exactly: {F.pt_eigenvalue_splitting(float(np.sqrt(0.99))):.12f}")
print("OK - ep_proximity_enclosure holds, and is TIGHT at the endpoint.")

  g = 0.0   splitting = 2.000000
  g = 0.5   splitting = 1.732051
  g = 0.99  splitting = 0.282135
  g = 1.0   splitting = 0.000000

max splitting on 99/100 <= g^2 <= 1 : 0.200000000000   [Lean bound: <= 1/5]
attained at g^2 = 99/100 exactly: 0.200000000000
OK - ep_proximity_enclosure holds, and is TIGHT at the endpoint.


In [6]:
# viz-ref: fig_d11_pt_exceptional_point
V.fig_d11_pt_exceptional_point().show()

## 3. The Haldane Chern witness (Phase 6ED) — and its retracted classification

Lean certifies `haldaneFrame_latticeChern_eq_neg_one`: **C = −1** at `t = t₂ = 1`, `φ = π/2`,
`m = 1`, on a 4×4 torus. The sign follows from the orientation convention
(`latticeChern := −∑ plaquetteBranch`).

The important companion is negative. `haldane_massInversion_not_sufficient_at_N4` gives **C = 0 at
m = 5**, which is *inside* the analytic mass-inversion window `|m| < 3√3 ≈ 5.196`. A theorem
asserting "nonzero Chern exactly where the masses invert" was **deleted** as both false at fixed N
and contentless.

In [7]:
window = F.haldane_mass_inversion_window(1.0, np.pi/2)
print(f"analytic inversion window half-width |3sqrt(3) t2 sin(phi)| = {window:.4f}   [3*sqrt(3) = {3*np.sqrt(3):.4f}]")

print("\n  m     mass(K)    mass(K')   inverted?   certified C (4x4)")
for m, C in ((1.0, -1), (5.0, 0), (6.0, 0)):
    mk, mkp = F.haldane_dirac_masses(m, 1.0, np.pi/2)
    inv = abs(m) < window
    print(f" {m:>4}  {mk:>9.4f}  {mkp:>9.4f}   {str(inv):<9}   {C:>+d}")

print("\nNOTE m = 5 is INSIDE the window yet certifies C = 0.")
print("Mass inversion is necessary but NOT sufficient at fixed grid size.")
assert abs(5.0) < window   # inside the analytic window ...
                           # ... yet the certified 4x4 invariant is 0.

analytic inversion window half-width |3sqrt(3) t2 sin(phi)| = 5.1962   [3*sqrt(3) = 5.1962]

  m     mass(K)    mass(K')   inverted?   certified C (4x4)
  1.0    -4.1962     6.1962   True        -1
  5.0    -0.1962    10.1962   True        +0
  6.0     0.8038    11.1962   False       +0

NOTE m = 5 is INSIDE the window yet certifies C = 0.
Mass inversion is necessary but NOT sufficient at fixed grid size.


In [8]:
# viz-ref: fig_d11_haldane_chern
V.fig_d11_haldane_chern().show()

## 4. Algebraic effective medium (Phase 6CE)

`effectiveMedium_hashinShtrikman_enclosure` certifies `(ε_h, ε_i, f) = (1, 4, 1/2) ⟹ ε_eff = 2`,
inside the constituent bounds. `voigt_sub_reuss_eq` gives the arithmetic-minus-harmonic gap
**exactly**, not as an inequality.

This layer is **algebraic-path only**: the two-scale homogenization route was not attempted. That is
a choice between two equally *unformalized* routes — two-scale convergence exists in no proof
assistant — not the avoidance of an existing formalization.

In [9]:
eps = F.maxwell_garnett(1.0, 4.0, 0.5)
print(f"Maxwell-Garnett (1, 4, 1/2) = {eps:g}   [Lean: exactly 2]")
assert abs(eps - 2.0) < 1e-12

fs = np.linspace(0.0, 1.0, 501)
vals = np.array([F.maxwell_garnett(1.0, 4.0, f) for f in fs])
print(f"constituent bounds respected over f in [0,1]: {bool((vals >= 1-1e-12).all() and (vals <= 4+1e-12).all())}")

# exact Voigt-Reuss gap
worst = 0.0
for f in fs:
    lhs = F.voigt_modulus(1.0, 4.0, f) - F.reuss_modulus(1.0, 4.0, f)
    worst = max(worst, abs(lhs - F.voigt_reuss_gap(1.0, 4.0, f)))
print(f"max |(M_V - M_R) - exact gap formula| over f : {worst:.3e}   [Lean voigt_sub_reuss_eq: identity]")
assert worst < 1e-12
print("OK - the gap is an exact identity, not merely non-negative.")

Maxwell-Garnett (1, 4, 1/2) = 2   [Lean: exactly 2]
constituent bounds respected over f in [0,1]: True
max |(M_V - M_R) - exact gap formula| over f : 5.551e-16   [Lean voigt_sub_reuss_eq: identity]
OK - the gap is an exact identity, not merely non-negative.


In [10]:
# viz-ref: fig_d11_effective_medium
V.fig_d11_effective_medium().show()

## 5. Scope — what is *not* here

Stated so this notebook cannot be read as claiming more than the substrate does:

- **Berry curvature is not formalized.** No Berry connection or curvature occurs anywhere in the
  development.
- **Bulk–boundary correspondence is deferred**, not shipped conditionally on a tracked hypothesis.
- **The continuum Chern integral `C = (1/2π)∫F` is deferred** — Mathlib has no manifold
  form-integration, no Stokes, no de Rham, no Brouwer degree.
- **Grids with `3 | N` are inadmissible** — they sample the Dirac points where the gauge
  representative degenerates.

The bundle carries **zero project-local axioms and zero tracked-hypothesis Props**: every result is
unconditional given its explicit mathematical hypotheses.